# 🧪 Exploration des données (Couche Silver)

Ce notebook permet de visualiser et d'explorer les données nettoyées dans la couche Silver.
**Objectifs :** 
- Vérifier la normalisation des noms de colonnes (snake_case).
- Valider le typage des données (Double pour les pourcentages).
- Utiliser des noms explicites (**pourcentage_...**) pour une meilleure sémantique métier.

## Session spark

In [ ]:
import os
from src.common.spark_session_manager import get_spark_session
from src.config import BRONZE_PATH, SILVER_PATH
from pyspark.sql.functions import col

# 1. Initialiser la session Spark
spark = get_spark_session(app_name="Exploration_Silver")

# 2. Configuration "Look Databricks" (Eager Evaluation)
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 20)

In [ ]:
nvp_2017_silver_path = os.path.join(SILVER_PATH, "niveau_vie_pauvrete", "niveau_vie_pauvrete_2017")
df_silver_2017 = spark.read.parquet(nvp_2017_silver_path)

In [ ]:
df_silver_2017.show(truncate=False)

In [ ]:
from pyspark.sql.functions import when

df_silver_2017 = df_silver_2017.withColumn(
    "est_limitrophe",
    when(
        (col("Arrondissement").isNotNull()) &
        (
            col("lcog_geo_2").isNotNull() |
            col("lcog_geo_3").isNotNull() |
            col("lcog_geo_4").isNotNull() |
            col("lcog_geo_5").isNotNull()
        ),
        1
    ).otherwise(0)
)

df_silver_2017.select(
    "Arrondissement", "lcog_geo_2", "lcog_geo_3", "lcog_geo_4", "lcog_geo_5", "est_limitrophe"
).show(20, truncate=False)

In [ ]:
df_silver_2017.show(truncate=False)